# Exercício 05 — Regularização e Overfitting
### Mastering Machine Learning Advanced · ML Engineer Track
### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

---

## Por que isso importa para um ML Engineer?

Overfitting é o problema mais comum em ML real. Saber **diagnosticá-lo e corrigi-lo** é fundamental para construir modelos que generalizam. Neste exercício você vai:

- Visualizar e medir overfitting com métricas de treino vs. teste
- Implementar regularização L1 (Lasso) e L2 (Ridge) do zero
- Entender como Dropout funciona em redes neurais
- Aplicar Early Stopping como técnica de regularização implícita
- Comparar o efeito de cada técnica na curva de aprendizado

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
np.random.seed(42)
%matplotlib inline

# dataset com muitas features para forçar overfitting
X_raw, y_raw = make_classification(
    n_samples=400, n_features=30, n_informative=5,
    n_redundant=10, n_repeated=5, random_state=42
)
scaler = StandardScaler()
X_raw = scaler.fit_transform(X_raw)

X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y_raw, test_size=0.25, random_state=42)
y_tr = y_tr.reshape(-1, 1)
y_te = y_te.reshape(-1, 1)

print(f'Treino: {X_tr.shape} | Teste: {X_te.shape}')
print(f'Features informativas: 5 de 30 — cenário favorável ao overfitting')

---
## Exercício 5.1 — Diagnosticando Overfitting

Antes de regularizar, você precisa **medir** o overfitting. Implemente uma função que treina uma rede por N épocas e retorna o histórico de loss de treino e validação.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

def bce(y_true, y_pred, eps=1e-9):
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


def treinar_com_historico(X_tr, y_tr, X_val, y_val,
                          n_hidden=64, lr=0.01, epochs=500,
                          lambda_l2=0.0, lambda_l1=0.0, dropout_rate=0.0,
                          early_stopping_patience=0):
    """
    Rede 2 camadas com suporte a L1, L2, Dropout e Early Stopping.
    Retorna: (params, hist_treino, hist_val)
    """
    n_input = X_tr.shape[1]
    W1 = np.random.randn(n_input, n_hidden) * np.sqrt(2 / n_input)
    b1 = np.zeros((1, n_hidden))
    W2 = np.random.randn(n_hidden, 1) * np.sqrt(2 / n_hidden)
    b2 = np.zeros((1, 1))

    hist_tr, hist_val = [], []
    best_val_loss = np.inf
    patience_counter = 0
    best_params = None

    for epoch in range(epochs):
        # --- forward ---
        Z1 = X_tr @ W1 + b1
        A1_raw = relu(Z1)

        # SEU CÓDIGO AQUI (5.2): aplique Dropout em A1_raw durante o treino
        # Por enquanto deixe A1 = A1_raw
        A1 = A1_raw
        mask = np.ones_like(A1)  # substituir no 5.2

        Z2 = A1 @ W2 + b2
        A2 = sigmoid(Z2)

        # --- loss com regularização ---
        loss_data = bce(y_tr, A2)

        # SEU CÓDIGO AQUI (5.1): adicione o termo de regularização ao loss
        # L2: (lambda_l2 / 2) * (||W1||² + ||W2||²)
        # L1: lambda_l1 * (||W1||_1 + ||W2||_1)
        reg_term = 0.0  # substitua pela implementação
        loss = loss_data + reg_term

        hist_tr.append(loss_data)  # log só o data loss para comparação justa

        # --- backward ---
        n = len(y_tr)
        dA2 = -(y_tr / (A2 + 1e-9)) + (1 - y_tr) / (1 - A2 + 1e-9)
        dZ2 = dA2 * A2 * (1 - A2)
        dW2 = A1.T @ dZ2 / n
        db2 = np.mean(dZ2, axis=0, keepdims=True)
        dA1 = dZ2 @ W2.T
        dA1 = dA1 * mask  # gradiente para o dropout
        dZ1 = dA1 * relu_deriv(Z1)
        dW1 = X_tr.T @ dZ1 / n
        db1 = np.mean(dZ1, axis=0, keepdims=True)

        # SEU CÓDIGO AQUI (5.1): adicione gradiente da regularização a dW1 e dW2
        # L2: dW += lambda_l2 * W
        # L1: dW += lambda_l1 * sign(W)

        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2

        # --- validação ---
        A1_val = relu(X_val @ W1 + b1)
        A2_val = sigmoid(A1_val @ W2 + b2)
        val_loss = bce(y_val, A2_val)
        hist_val.append(val_loss)

        # SEU CÓDIGO AQUI (5.3): Early Stopping
        # se early_stopping_patience > 0:
        #   monitore val_loss; se não melhorar por `patience` épocas, pare
        #   restaure os melhores pesos antes de retornar

    params = {'W1': W1, 'b1': b1, 'W2': W2, 'b2': b2}
    return params, hist_tr, hist_val


def overfitting_gap(hist_tr, hist_val, n_ultimas=50):
    """
    Métrica simples de overfitting: diferença média entre val e treino
    nas últimas N épocas.
    """
    # SEU CÓDIGO AQUI
    # retorne mean(hist_val[-n:]) - mean(hist_tr[-n:])
    pass


# --- VALIDAÇÃO ---
np.random.seed(42)
params_base, hist_tr_base, hist_val_base = treinar_com_historico(
    X_tr, y_tr, X_te, y_te, n_hidden=64, lr=0.01, epochs=300
)

gap = overfitting_gap(hist_tr_base, hist_val_base)
print(f'Overfitting gap (sem regularização): {gap:.4f}')
assert gap is not None, 'overfitting_gap deve retornar um valor'
assert hist_val_base[-1] > hist_tr_base[-1], 'Esperado: val loss > train loss (overfitting)'
print('✅ Exercício 5.1 correto!')

---
## Exercício 5.2 — Regularização L1 e L2

Implemente os termos de regularização dentro de `treinar_com_historico` (células acima, marcadas com `SEU CÓDIGO AQUI (5.1)`).

**L2 (Ridge / Weight Decay):**
- Adiciona `(λ/2) * ||W||²` ao loss
- Gradiente extra: `λ * W`
- Penaliza pesos grandes → pesos ficam pequenos e distribuídos

**L1 (Lasso):**
- Adiciona `λ * ||W||₁` ao loss
- Gradiente extra: `λ * sign(W)`
- Produz sparsidade — força alguns pesos a zero

In [ ]:
# --- VALIDAÇÃO: L2 ---
np.random.seed(42)
_, hist_tr_l2, hist_val_l2 = treinar_com_historico(
    X_tr, y_tr, X_te, y_te, n_hidden=64, lr=0.01, epochs=300, lambda_l2=0.01
)

gap_l2 = overfitting_gap(hist_tr_l2, hist_val_l2)
print(f'Overfitting gap — sem reg: {overfitting_gap(hist_tr_base, hist_val_base):.4f}')
print(f'Overfitting gap — L2:      {gap_l2:.4f}')
assert gap_l2 < overfitting_gap(hist_tr_base, hist_val_base), \
    'L2 deve reduzir o overfitting gap'

# --- VALIDAÇÃO: L1 ---
np.random.seed(42)
_, hist_tr_l1, hist_val_l1 = treinar_com_historico(
    X_tr, y_tr, X_te, y_te, n_hidden=64, lr=0.01, epochs=300, lambda_l1=0.001
)

gap_l1 = overfitting_gap(hist_tr_l1, hist_val_l1)
print(f'Overfitting gap — L1:      {gap_l1:.4f}')
assert gap_l1 < overfitting_gap(hist_tr_base, hist_val_base), \
    'L1 deve reduzir o overfitting gap'

print('✅ Exercício 5.2 correto!')

---
## Exercício 5.3 — Dropout

Dropout desativa aleatoriamente `p` fração dos neurônios a cada forward pass durante o treino, forçando a rede a aprender representações redundantes e robustas.

**Implementação (inverted dropout):**
- Gere máscara binária: `mask = (np.random.rand(*A.shape) > dropout_rate)`
- Aplique: `A_dropped = A * mask / (1 - dropout_rate)` ← divisão compensa a escala
- No backward, multiplique o gradiente pela mesma máscara
- Em inferência (validação): use `A` sem dropout

Implemente dentro de `treinar_com_historico` onde está marcado `SEU CÓDIGO AQUI (5.2)`.

In [ ]:
# --- VALIDAÇÃO: Dropout ---
np.random.seed(42)
_, hist_tr_do, hist_val_do = treinar_com_historico(
    X_tr, y_tr, X_te, y_te, n_hidden=64, lr=0.01, epochs=300, dropout_rate=0.3
)

gap_do = overfitting_gap(hist_tr_do, hist_val_do)
print(f'Overfitting gap — Dropout: {gap_do:.4f}')
assert gap_do < overfitting_gap(hist_tr_base, hist_val_base), \
    'Dropout deve reduzir o overfitting gap'
print('✅ Exercício 5.3 correto!')

---
## Exercício 5.4 — Early Stopping

Early Stopping interrompe o treino quando o val loss para de melhorar, salvando os pesos do melhor momento.

Implemente dentro de `treinar_com_historico` onde está marcado `SEU CÓDIGO AQUI (5.3)`:
- Se `early_stopping_patience > 0`, monitore `val_loss` a cada época
- Se o val loss não melhorar por `patience` épocas consecutivas, pare
- Restaure `best_params` antes de retornar

In [ ]:
# --- VALIDAÇÃO: Early Stopping ---
np.random.seed(42)
params_es, hist_tr_es, hist_val_es = treinar_com_historico(
    X_tr, y_tr, X_te, y_te, n_hidden=64, lr=0.01, epochs=500,
    early_stopping_patience=20
)

print(f'Épocas treinadas (early stopping): {len(hist_tr_es)} de 500')
assert len(hist_tr_es) < 500, 'Early stopping deve parar antes de 500 épocas'
assert len(hist_tr_es) > 20, 'Deve treinar ao menos patience épocas'

gap_es = overfitting_gap(hist_tr_es, hist_val_es)
print(f'Overfitting gap — Early Stopping: {gap_es:.4f}')
print('✅ Exercício 5.4 correto!')

---
## Exercício 5.5 — Comparativo Visual

Plote as curvas de validação de todos os experimentos no mesmo gráfico.

In [ ]:
# SEU CÓDIGO AQUI
# Plote hist_val_base, hist_val_l2, hist_val_l1, hist_val_do, hist_val_es
# Use cores e legendas distintas
# Título: 'Efeito da Regularização no Val Loss'
# Qual técnica deu o menor val loss final? Qual reduziu mais o overfitting gap?

**Reflexão:** quando você usaria L1 vs. L2 vs. Dropout? Existe alguma situação onde combinar as três faz sentido?

*(Escreva sua resposta aqui)*

---
**Prof. Dr. Ahirton Lopes** | [LinkedIn](https://linkedin.com/in/ahirtonlopes) | [GitHub](https://github.com/ahirtonlopes)